# P4A Execution Control Gate
**NON-RESULT-BEARING.** This notebook validates Drive/Git/environment/control evidence only. It does not authorize or execute P4B scientific tasks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('DRIVE_MOUNT=PASS')


In [ ]:
from google.colab import userdata
from pathlib import Path
import os, subprocess, sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'P4A requires Python 3.12; got {sys.version.split()[0]}')
REPO = 'ArashSalehpourac/BreastCancer-Imbalance-Benchmark_35'
AUTHORIZED_COMMIT = '<SEPARATELY_REVIEWED_EXACT_P4A_COMMIT_SHA>'
DATASET = '/content/drive/MyDrive/35/01_Experiment_Evidence/00_Dataset/wdbc_canonical_lf.csv'
CONTROL_BASE = '/content/drive/MyDrive/35/01_Experiment_Evidence/P4A_Control_Validation'
TOKEN = userdata.get('GITHUB_TOKEN')
if not TOKEN:
    raise RuntimeError('GITHUB_TOKEN Colab secret is required')
if len(AUTHORIZED_COMMIT) != 40 or '<' in AUTHORIZED_COMMIT:
    raise RuntimeError('Set AUTHORIZED_COMMIT to the separately reviewed exact 40-character SHA')
askpass = Path('/tmp/p4a_git_askpass.sh')
askpass.write_text("""#!/bin/sh
case \"$1\" in
  *Username*) printf '%s\\n' 'x-access-token' ;;
  *Password*) printf '%s\\n' \"$GITHUB_TOKEN\" ;;
  *) printf '%s\\n' \"$GITHUB_TOKEN\" ;;
esac
""")
askpass.chmod(0o700)
env = os.environ.copy()
env['GITHUB_TOKEN'] = TOKEN
env['GIT_ASKPASS'] = str(askpass)
env['GIT_TERMINAL_PROMPT'] = '0'
repo_dir = Path('/content/BreastCancer-Imbalance-Benchmark_35')
if repo_dir.exists():
    subprocess.run(['rm','-rf',str(repo_dir)], check=True)
subprocess.run(['git','clone',f'https://github.com/{REPO}.git',str(repo_dir)], check=True, env=env)
subprocess.run(['git','-C',str(repo_dir),'checkout','--detach',AUTHORIZED_COMMIT], check=True, env=env)
head = subprocess.check_output(['git','-C',str(repo_dir),'rev-parse','HEAD'], text=True).strip()
if head != AUTHORIZED_COMMIT:
    raise RuntimeError('Exact Git identity gate failed')
print('P4A_REPOSITORY_IDENTITY=PASS')


In [ ]:
subprocess.run(['python','-m','pip','install','-r',str(repo_dir/'requirements/p3-preflight.txt')], check=True)
subprocess.run(['python','-m','pip','install','-e',str(repo_dir),'--no-deps'], check=True)
print('P4A_ENVIRONMENT_INSTALL=PASS')


In [ ]:
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
session = f'P4A_CONTROL_{stamp}_{AUTHORIZED_COMMIT[:12]}'
control_root = f'{CONTROL_BASE}/{session}'
cmd = [
    'python', str(repo_dir/'scripts/p4a_control.py'),
    '--dataset', DATASET,
    '--repo-root', str(repo_dir),
    '--expected-git-commit', AUTHORIZED_COMMIT,
    '--control-root', control_root,
    '--policy', str(repo_dir/'config/P4A_EXECUTION_CONTROL_POLICY_v1.json'),
    '--config', str(repo_dir/'config/EXPERIMENT_CONFIG_v1.json'),
    '--foundation-lock', str(repo_dir/'data/registry/FOUNDATION_LOCK_v1.json'),
    '--result-schema', str(repo_dir/'config/RESULT_EVIDENCE_SCHEMA_v1.json'),
    '--authorization-ref', 'GitHub Issue #11 P4A Colab validation authorization',
    '--require-exact-colab-paths',
]
subprocess.run(cmd, check=True)
print('P4A_COLAB_CONTROL_COMPLETE=PASS')
print('P4B_AUTHORIZED=false')
print('RESULT_BEARING=false')
print('SCIENTIFIC_EXECUTION=0')
